# TP — Module 1 : Fondements Python pour l'Analyse Géospatiale
## Exploration des données du Burkina Faso

> **Cours** : Analyse Spatiale avec Machine Learning (ASML)  
> **Institut** : 2iE — Master Eau, Environnement, Aménagement  
> **Durée** : 6h (CM 1h + TP 2h + travail guidé)  
> **Prérequis** : Python intermédiaire (Pandas, NumPy)

---

## Objectifs

Ce TP applique directement le cours CM sur les données du Burkina Faso. À l'issue :

| # | Compétence | Section CM |
|---|-----------|------------|
| 1 | Charger et explorer un GeoDataFrame | §3.1–3.3 |
| 2 | Calculer superficie, périmètre, distance (CRS métrique) | §3.3 + §5 |
| 3 | Réaliser une jointure spatiale (`sjoin`) | §3.4 |
| 4 | Ouvrir un raster et calculer le NDVI | §4 |
| 5 | Produire une carte statique avec fond Contextily | §6.1–6.2 |
| 6 | Créer une carte interactive Folium | §6.3 |

## Données disponibles

- **13 régions administratives** du Burkina Faso (GeoJSON embarqué)
- **30 points d'eau** (forages et puits, GeoJSON embarqué)
- **Raster NDVI synthétique** (généré en NumPy — reproduit le gradient sahélien réel)

## Environnement

> **Google Colab** : décommenter les lignes `!pip install` ci-dessous.

In [ ]:
# Décommenter si nécessaire (Google Colab)
# !pip install geopandas rasterio folium contextily matplotlib --quiet

### Imports

In [ ]:
import json, io, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import rasterio
from rasterio.transform import from_bounds
from rasterstats import zonal_stats
import folium
import contextily as ctx
import requests

print('Bibliothèques chargées. GeoPandas :', gpd.__version__)

# ═══════════════════════════════════════════════════════════════
# Données fallback — définies EN PREMIER (évite NameError)
# ═══════════════════════════════════════════════════════════════

GJ_REG_STR = '{"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"region": "Boucle du Mouhoun", "chef_lieu": "Dédougou", "population": 1427000, "superficie_ref": 34497}, "geometry": {"type": "Polygon", "coordinates": [[[-3.45, 12.15], [-2.3499999999999996, 12.15], [-2.3499999999999996, 13.049999999999999], [-3.45, 13.049999999999999], [-3.45, 12.15]]]}}, {"type": "Feature", "properties": {"region": "Cascades", "chef_lieu": "Banfora", "population": 730000, "superficie_ref": 18236}, "geometry": {"type": "Polygon", "coordinates": [[[-5.0, 10.05], [-3.9000000000000004, 10.05], [-3.9000000000000004, 10.95], [-5.0, 10.95], [-5.0, 10.05]]]}}, {"type": "Feature", "properties": {"region": "Centre", "chef_lieu": "Ouagadougou", "population": 2415000, "superficie_ref": 2585}, "geometry": {"type": "Polygon", "coordinates": [[[-2.0700000000000003, 11.91], [-0.97, 11.91], [-0.97, 12.809999999999999], [-2.0700000000000003, 12.809999999999999], [-2.0700000000000003, 11.91]]]}}, {"type": "Feature", "properties": {"region": "Centre-Est", "chef_lieu": "Tenkodogo", "population": 1430000, "superficie_ref": 14442}, "geometry": {"type": "Polygon", "coordinates": [[[-0.35000000000000003, 11.4], [0.75, 11.4], [0.75, 12.299999999999999], [-0.35000000000000003, 12.299999999999999], [-0.35000000000000003, 11.4]]]}}, {"type": "Feature", "properties": {"region": "Centre-Nord", "chef_lieu": "Kaya", "population": 1400000, "superficie_ref": 19889}, "geometry": {"type": "Polygon", "coordinates": [[[-1.6, 12.600000000000001], [-0.5, 12.600000000000001], [-0.5, 13.5], [-1.6, 13.5], [-1.6, 12.600000000000001]]]}}, {"type": "Feature", "properties": {"region": "Centre-Ouest", "chef_lieu": "Koudougou", "population": 1360000, "superficie_ref": 21543}, "geometry": {"type": "Polygon", "coordinates": [[[-3.0, 11.65], [-1.9000000000000001, 11.65], [-1.9000000000000001, 12.549999999999999], [-3.0, 12.549999999999999], [-3.0, 11.65]]]}}, {"type": "Feature", "properties": {"region": "Centre-Sud", "chef_lieu": "Manga", "population": 730000, "superficie_ref": 11731}, "geometry": {"type": "Polygon", "coordinates": [[[-1.6, 11.05], [-0.5, 11.05], [-0.5, 11.95], [-1.6, 11.95], [-1.6, 11.05]]]}}, {"type": "Feature", "properties": {"region": "Est", "chef_lieu": "Fada N\'Gourma", "population": 1580000, "superficie_ref": 47245}, "geometry": {"type": "Polygon", "coordinates": [[[0.8, 11.65], [1.9000000000000001, 11.65], [1.9000000000000001, 12.549999999999999], [0.8, 12.549999999999999], [0.8, 11.65]]]}}, {"type": "Feature", "properties": {"region": "Hauts-Bassins", "chef_lieu": "Bobo-Dioulasso", "population": 2040000, "superficie_ref": 25739}, "geometry": {"type": "Polygon", "coordinates": [[[-4.8, 10.700000000000001], [-3.7, 10.700000000000001], [-3.7, 11.6], [-4.8, 11.6], [-4.8, 10.700000000000001]]]}}, {"type": "Feature", "properties": {"region": "Nord", "chef_lieu": "Ouahigouya", "population": 1110000, "superficie_ref": 16855}, "geometry": {"type": "Polygon", "coordinates": [[[-2.9699999999999998, 13.13], [-1.8699999999999999, 13.13], [-1.8699999999999999, 14.03], [-2.9699999999999998, 14.03], [-2.9699999999999998, 13.13]]]}}, {"type": "Feature", "properties": {"region": "Plateau Central", "chef_lieu": "Ziniaré", "population": 790000, "superficie_ref": 8018}, "geometry": {"type": "Polygon", "coordinates": [[[-1.17, 12.100000000000001], [-0.06999999999999995, 12.100000000000001], [-0.06999999999999995, 13.0], [-1.17, 13.0], [-1.17, 12.100000000000001]]]}}, {"type": "Feature", "properties": {"region": "Sahel", "chef_lieu": "Dori", "population": 1230000, "superficie_ref": 44290}, "geometry": {"type": "Polygon", "coordinates": [[[-0.6000000000000001, 13.600000000000001], [0.5, 13.600000000000001], [0.5, 14.5], [-0.6000000000000001, 14.5], [-0.6000000000000001, 13.600000000000001]]]}}, {"type": "Feature", "properties": {"region": "Sud-Ouest", "chef_lieu": "Diébougou", "population": 730000, "superficie_ref": 16126}, "geometry": {"type": "Polygon", "coordinates": [[[-3.8, 10.15], [-2.7, 10.15], [-2.7, 11.049999999999999], [-3.8, 11.049999999999999], [-3.8, 10.15]]]}}]}'

fallback_gdf = gpd.GeoDataFrame.from_features(
    json.loads(GJ_REG_STR)['features'], crs='EPSG:4326'
)

GJ_EAU_STR = 'None'

puits = gpd.GeoDataFrame.from_features(
    json.loads(GJ_EAU_STR)['features'], crs='EPSG:4326'
)

print(f'Fallback : {len(fallback_gdf)} régions + {len(puits)} points d\'eau')

# ═══════════════════════════════════════════════════════════════
# Chargement GADM 4.1 — 13 vraies régions BF
# Source : https://gadm.org/download_country.html → GeoJSON level-1
# URL    : geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BFA_1.json
# Fallback automatique si GADM inaccessible
# ═══════════════════════════════════════════════════════════════

URL_GADM_L1 = 'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BFA_1.json'

def load_gadm_regions(fallback):
    try:
        print('Chargement GADM level-1 (vraies géométries BF)...')
        r = requests.get(URL_GADM_L1, timeout=15)
        r.raise_for_status()
        gdf_raw = gpd.read_file(io.BytesIO(r.content))
        gdf_raw = gdf_raw.rename(columns={
            'NAME_0':'pays', 'NAME_1':'region', 'GID_1':'gid_region'
        })
        for col in ['chef_lieu','population','superficie_ref']:
            if col in fallback.columns and col not in gdf_raw.columns:
                mapping = fallback.set_index('region')[col].to_dict()
                gdf_raw[col] = gdf_raw['region'].map(mapping)
        print(f'[GADM ✅] {len(gdf_raw)} régions | CRS: {gdf_raw.crs}')
        return gdf_raw, 'GADM 4.1'
    except Exception as e:
        warnings.warn(f'GADM inaccessible ({e}) → fallback', stacklevel=2)
        print('[FALLBACK ⚠️] Rectangles embarqués utilisés')
        return fallback.copy(), 'fallback'

gdf, source_gdf = load_gadm_regions(fallback_gdf)

print(f'\nSource  : {source_gdf}')
print(f'Régions : {len(gdf)} | CRS : {gdf.crs}')


---
## Partie 1 — GeoDataFrame et Exploration des Données

> **Référence CM** : §3.1 Le GeoDataFrame, §3.2 Lecture des fichiers spatiaux

### 1.1 Chargement des données

Les deux jeux de données utilisés dans ce TP sont embarqués directement :
- `gdf` : 13 régions administratives du Burkina Faso (polygones)
- `puits` : 30 points d'eau — forages et puits AEPS/PEA (points)

### 1.2 Exercice 1 — Exploration

En suivant le pattern du CM (§3.1), répondez aux questions suivantes :

1. Affichez les **3 premières lignes** du GeoDataFrame `gdf`
2. Affichez les **types de colonnes** (`dtypes`)
3. Vérifiez que le type géométrique est bien **Polygon** pour toutes les régions
4. Quelle région a la **population la plus élevée** ? La plus faible ?
5. Combien y a-t-il de **forages** et de **puits** dans `puits` ?

In [ ]:
# ── 1. Afficher les 3 premières lignes ──
# Votre code ici


# ── 2. Types de colonnes ──
# Votre code ici


# ── 3. Type géométrique ──
# Votre code ici


# ── 4. Région la plus et moins peuplée ──
# Votre code ici


# ── 5. Décompte forages vs puits ──
# Votre code ici

---
## Partie 2 — Attributs Géométriques et Projections

> **Référence CM** : §3.3 Attributs géométriques, §5 Systèmes de référence

Le CM insiste sur une **règle absolue** :
> *Vérifiez toujours le CRS avant de calculer une superficie ou une distance.*
> *En EPSG:4326 (degrés), `.area` donne des degrés carrés — sans sens physique.*

### Exercice 2 — Métriques géométriques

1. Montrez que `gdf.geometry.area.mean()` en EPSG:4326 donne une valeur absurde
2. Reprojetez en **EPSG:32630** (UTM Zone 30N) et calculez :
   - `superficie_km2` (km²)
   - `perimetre_km` (km)
3. Calculez la **distance de chaque région au centroïde d'Ouagadougou** (en km)
   - Ouagadougou : lon = -1.5243, lat = 12.3647
4. Comparez `superficie_km2` calculée avec `superficie_ref` (données officielles INSD) :
   quel est l'écart moyen en % ? Expliquez pourquoi.

In [ ]:
# ── 2.1 : Superficie en degrés² (absurde) ──
# Votre code ici


# ── 2.2 : Reprojection + superficie + périmètre ──
# Votre code ici


# ── 2.3 : Distance à Ouagadougou ──
# Votre code ici


# ── 2.4 : Comparaison avec superficie officielle ──
# Votre code ici

---
## Partie 3 — Jointures Spatiales

> **Référence CM** : §3.4 Jointures spatiales — `gpd.sjoin`

Le CM présente le cas : *quelle région contient chaque point d'eau ?*
C'est exactement ce que vous allez réaliser avec nos 30 points d'eau.

### Exercice 3 — Analyser la couverture en eau potable

1. Vérifiez que `gdf` et `puits` ont le **même CRS** avant la jointure
2. Réalisez une jointure spatiale `gpd.sjoin` pour savoir dans quelle région
   se trouve chaque point d'eau (`predicate='within'`)
3. Calculez le **nombre de points d'eau par région** et ajoutez la colonne
   `nb_points_eau` au GeoDataFrame `gdf`
4. Calculez le **ratio points d'eau / million d'habitants** (`ratio_peau_M`)
5. Identifiez la région la mieux et la moins bien équipée en eau potable

> ⚠️ Rappel CM : *les deux GeoDataFrames doivent être dans le même CRS*
> *avant une jointure spatiale — sinon les résultats sont silencieusement faux.*

In [ ]:
# ── 3.1 : Vérification CRS identiques ──
# Votre code ici


# ── 3.2 : Jointure spatiale (sjoin) ──
# Votre code ici


# ── 3.3 : Comptage par région et merge ──
# Votre code ici


# ── 3.4 : Ratio eau / million d'habitants ──
# Votre code ici


# ── 3.5 : Meilleures / moins bonnes régions ──
# Votre code ici

---
## Partie 4 — Données Raster et Calcul du NDVI

> **Référence CM** : §4.1 Ouvrir et lire un raster, §4.2 Calcul du NDVI

Dans un projet réel, vous chargeriez des images Sentinel-2 depuis Copernicus
ou Google Earth Engine. Dans ce TP, nous générons un raster **synthétique**
qui reproduit le gradient bioclimatique Nord-Sud du Burkina Faso :
zones arides au Nord (Sahel, NDVI ~ 0.05), zones humides au Sud (NDVI ~ 0.60).

### 4.1 Génération et écriture du raster synthétique

> **Rappel CM §4.1** : Rasterio gère les rasters comme un livre —
> on l'ouvre avec `with rasterio.open()`, on lit les bandes (indices commençant à 1),
> puis on le ferme automatiquement à la sortie du bloc `with`.

**Exercice 4a** — Exécutez la cellule ci-dessous pour créer `bf_ndvi_synth.tif`.
Lisez le code et répondez : pourquoi utilise-t-on `.astype(float)` ?

In [ ]:
import rasterio
from rasterio.transform import from_bounds

# ── Paramètres du raster synthétique ──
# Bbox du Burkina Faso : lon -5.5 → 2.4 | lat 9.4 → 15.1
BBOX    = (-5.5, 9.4, 2.4, 15.1)   # (west, south, east, north)
ROWS, COLS = 100, 130               # Résolution pédagogique
NODATA  = -9999.0

# ── Simulation du gradient bioclimatique Nord-Sud ──
# lat augmente vers le Nord → gradient décroissant du Sud (NDVI élevé) vers le Nord
lats = np.linspace(BBOX[3], BBOX[1], ROWS)   # du Nord (row 0) vers le Sud (row -1)
lons = np.linspace(BBOX[0], BBOX[2], COLS)
LON, LAT = np.meshgrid(lons, lats)

# Gradient latitudinal : lat 15 → NDVI 0.05 | lat 9.4 → NDVI 0.62
ndvi_base = 0.62 - (LAT - 9.4) * (0.62 - 0.05) / (15.1 - 9.4)

# Bruit spatial réaliste
np.random.seed(42)
bruit = np.random.normal(0, 0.04, (ROWS, COLS))
ndvi  = np.clip(ndvi_base + bruit, -0.1, 0.9).astype(np.float32)

# ── Écriture GeoTIFF via Rasterio ──
transform = from_bounds(*BBOX, width=COLS, height=ROWS)

profile = {
    'driver': 'GTiff',
    'dtype': rasterio.float32,
    'width': COLS,
    'height': ROWS,
    'count': 1,          # 1 seule bande (le NDVI)
    'crs': 'EPSG:4326',
    'transform': transform,
    'nodata': NODATA
}

RASTER_PATH = 'bf_ndvi_synth.tif'
with rasterio.open(RASTER_PATH, 'w', **profile) as dst:
    dst.write(ndvi, 1)   # bande 1 (indices commencent à 1, pas 0 !)

print('Raster créé :', RASTER_PATH)
print(f'  Dimensions : {ROWS} × {COLS} pixels')
print(f'  NDVI min   : {ndvi.min():.3f} | max : {ndvi.max():.3f} | mean : {ndvi.mean():.3f}')

### 4.2 Exercice 4b — Lecture et inspection du raster

En suivant le CM §4.1, utilisez `rasterio.open()` pour inspecter le raster créé :

1. Affichez : CRS, résolution (`res`), dimensions (`shape`), nombre de bandes (`count`), nodata
2. Lisez la bande 1 avec `.read(1)` — notez que l'indice commence à **1**
3. Calculez et affichez : min, max, moyenne du NDVI
4. Expliquez pourquoi `.astype(float)` est essentiel avant le calcul du NDVI
   *(Indice : que se passe-t-il si on divise deux entiers uint16 en Python/NumPy ?)*

In [ ]:
# ── 4b.1 : Inspection des métadonnées ──
# Votre code ici


# ── 4b.2 : Lecture de la bande 1 ──
# Votre code ici


# ── 4b.3 : Statistiques NDVI ──
# Votre code ici


# ── 4b.4 : Explication astype(float) ──
# Votre réponse (commentaire) :
# ...

### 4.3 Exercice 4c — NDVI moyen par région

Calculez le **NDVI moyen de chaque région** en utilisant `rasterstats.zonal_stats`.
Ajoutez la colonne `ndvi_mean` au GeoDataFrame `gdf`.

> 💡 `rasterstats` effectue automatiquement les statistiques zonales :
> pour chaque polygone de `gdf`, il agrège les pixels du raster qui tombent dedans.

In [ ]:
# Installation si nécessaire :
# !pip install rasterstats --quiet
from rasterstats import zonal_stats

# ── Exercice 4c : Statistiques zonales NDVI ──
# Votre code ici


# Afficher le résultat trié par NDVI croissant
# Votre code ici

---
## Partie 5 — Cartographie Statique avec Contextily

> **Référence CM** : §6.1 Carte rapide avec `.plot()`, §6.2 Fond de carte Contextily

Le CM présente deux niveaux de visualisation statique :
1. `.plot()` direct sur un GeoDataFrame (rapide, pour le diagnostic)
2. `contextily` pour ajouter un fond de carte OpenStreetMap (livrable professionnel)

> ⚠️ **Rappel CM §6.2** : Contextily requiert **EPSG:3857** (Web Mercator).
> Il faut reprojeter le GeoDataFrame avant d'appeler `ctx.add_basemap()`.

### Exercice 5 — Deux cartes côte à côte

Créez une figure avec deux axes :
1. **Axe gauche** : Choroplèthe de la **population** avec `scheme='quantiles'`, k=4
   - Titre, légende, `axis('off')`, annotations des noms de régions
2. **Axe droit** : Fond de carte Contextily + points d'eau colorés par type
   - Reprojeter en EPSG:3857 avant `ctx.add_basemap()`
   - Forages en bleu, Puits en orange

In [ ]:
import contextily as ctx

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Exercice 5.1 : Choroplèthe population (axes[0]) ──
# Votre code ici


# ── Annotations des noms de régions ──
# Votre code ici


# ── Exercice 5.2 : Contextily + points d'eau (axes[1]) ──
# Votre code ici


plt.tight_layout()
plt.savefig('M1_cartes_statiques.png', dpi=150, bbox_inches='tight')
plt.show()
print('Carte sauvegardée : M1_cartes_statiques.png')

---
## Partie 6 — Carte Interactive avec Folium

> **Référence CM** : §6.3 Carte interactive avec Folium

Le CM présente Folium comme l'outil de livrable web : *idéal pour des interfaces
destinées à des non-techniciens ou des décideurs*.

### Exercice 6 — Tableau de bord interactif

Créez une carte Folium avec **3 couches** :

1. **Choroplèthe** de la population par région (`folium.Choropleth`)
2. **Tooltips** sur les régions affichant : Région, Chef-lieu, Population,
   Nb points d'eau, NDVI moyen
3. **Marqueurs** pour les points d'eau — icône différente selon le type :
   - Forage : `icon='tint'` (bleu)
   - Puits : `icon='circle'` (orange)

> ⚠️ **Rappel CM** : Folium travaille toujours en **EPSG:4326** (WGS84).
> Ne jamais lui passer un GeoDataFrame en EPSG:32630 ou 3857.

> 💡 **Bonne pratique popup** : construire le HTML par concaténation de strings —
> jamais avec des f-strings triple-guillemets imbriqués dans du code notebook.

In [ ]:
# ── 6.1 : Initialisation ──
# Centre BF : lat=12.37, lon=-1.54
# Votre code ici


# ── 6.2 : Choroplèthe population ──
# Votre code ici


# ── 6.3 : Tooltips régions ──
# Votre code ici


# ── 6.4 : Marqueurs points d'eau ──
# Votre code ici


# ── 6.5 : Contrôle de couches + sauvegarde ──
# folium.LayerControl().add_to(m)
# m.save('M1_carte_interactive.html')
# print('Carte sauvegardée : M1_carte_interactive.html')
# m  # Affichage dans Jupyter

---
## ✅ Conclusion du TP — Module 1

Vous avez appliqué les 6 sections du CM sur des données réelles du Burkina Faso :

| Compétence | Outil | Section CM |
|-----------|-------|-----------|
| Chargement & exploration GeoDataFrame | `gpd.GeoDataFrame.from_features()` | §3.1–3.2 |
| Superficie, périmètre, distance (CRS métrique) | `.to_crs(32630)`, `.area`, `.distance()` | §3.3 + §5 |
| Jointure spatiale | `gpd.sjoin()` | §3.4 |
| Ouverture raster & NDVI | `rasterio.open()`, `rasterstats` | §4 |
| Carte statique + fond OSM | `.plot()` + `contextily` | §6.1–6.2 |
| Carte interactive | `folium.Choropleth` + `GeoJsonTooltip` | §6.3 |

---
**Module 2** : Feature Engineering Spatial — vous enrichirez ces données
avec des indices de voisinage (Moran, lag spatial) et des features spectrales.